# Anime Review Sentiment — Notebook 1: Data Scraping

Scrapes reviews and star ratings from MyAnimeList (MAL) for the 200 most popular anime, verifies the raw
data, and splits it into a development set and a held-out test set.

**At a glance**
- **19,723 reviews across 200 titles**, scraped politely (1 s delay, back-off, page cache, no usernames stored)
- **Labels come free:** each reviewer's own 1–10 rating becomes the sentiment label, so no manual annotation
- **Rating-stratified 95/5 split:** Dataset A (18,738) for development, Dataset B (985) locked away for the final test

**Why 200 titles instead of one?** A model trained on one show learns that show's vocabulary and fails on
others. Spreading the data across many titles is what lets it generalise to anime it has never seen.

| Output | Contents |
|---|---|
| `data/raw_reviews_combined.csv` | Every scraped review, unmodified |
| `data/Dataset_A_raw.csv` | ~95%, for training and validation |
| `data/Dataset_B_raw.csv` | ~5%, held out until the final evaluation |

This notebook only collects and verifies. All cleaning happens in notebook 2, so the "raw" files really are raw.

**Pipeline:** **1. Scraping** → 2. Data Preparation → 3. Modelling & Evaluation

> **Runtime & reproducibility.** The scraped data isn't included in this repository, so the first run
> scrapes the live site: ~33 minutes for this notebook, ~59 minutes for all three on a CPU. Pages are cached
> in `data/cache/`, so re-runs take under a minute. MAL changes over time, so a fresh run gives numbers close
> to, but not identical to, the saved outputs (from July 2026).
>
> **Quicker run:** lower `N_TITLES` (e.g. 100) and/or `MAX_REVIEW_PAGES` (e.g. 1) in the config cell below.
> Notebooks 2 and 3 adapt automatically: notebook 2's name-removal threshold is a *proportion* of the titles
> (15%), not a fixed count, so it stays correct with fewer titles. Expect weaker results, since fewer titles
> means less vocabulary variety. See the README for timings.

## Target variable

`sentiment` (binary) is derived in notebook 2 from the reviewer's own star rating, scraped here alongside
the text:

| Star rating | `sentiment` |
|---|---|
| 7–10 | 1 (positive) |
| 5–6 | dropped — usually mixed |
| 1–4 | 0 (negative) |

Ratings stay on the raw 1–10 scale here, so the thresholds can be changed later without re-scraping.

## 1. Scraping ethics

MAL's `robots.txt` was checked first (July 2026). Review pages are not disallowed; only admin and private
paths and a list of named AI crawlers (`GPTBot`, `ClaudeBot`, `CCBot`, `Scrapy`) are blocked, none of which
applies to this small supervised classifier.

| Concern | Measure |
|---|---|
| Server load | 1 s delay + random jitter between requests (under 1 request/second sustained) |
| Rate limiting | Increasing back-off on HTTP 403 and 429 (6 s, then 12 s, then 18 s) |
| Redundant traffic | Local disk cache — no page is downloaded twice |
| Privacy | Usernames, avatars and profile links are never saved; only text, rating, verdict tag and date |
| Terms of use | Public pages only, no login |

The delay was originally planned at 3 s and lowered to 1 s to match common public scrapers for this site;
the back-off remains the safety net.

In [ ]:
# ============================================================
# Imports & global configuration
# ============================================================
import os
import re
import time
import random
import hashlib

import requests
from bs4 import BeautifulSoup
import pandas as pd

# ---- scraping configuration -------------------------------------------------
BASE_URL = "https://myanimelist.net"

# Browser-like headers: MAL sits behind an Imperva/Incapsula WAF which serves
# an interstitial/403 to "empty" clients. These headers make the WAF serve the
# same public page a normal visitor sees.
HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) "
                   "Chrome/126.0.0.0 Safari/537.36"),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://myanimelist.net/",
}

REQUEST_DELAY   = 1      # base politeness delay in seconds (see rate note in section 1)
N_TITLES        = 200    # how many popular titles to harvest reviews from
MAX_REVIEW_PAGES = 5    # review pages per title (20 reviews per page -> up to 100/title)
RANDOM_STATE    = 42     # reproducibility for the 95/5 split

DATA_DIR  = "data"
CACHE_DIR = os.path.join(DATA_DIR, "cache")   # fetched pages are cached here
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"Config: {N_TITLES} titles x up to {MAX_REVIEW_PAGES} pages "
      f"(~{N_TITLES * MAX_REVIEW_PAGES} requests, ~{REQUEST_DELAY}s apart)")
print(f"Estimated runtime: ~{N_TITLES * MAX_REVIEW_PAGES * (REQUEST_DELAY + 1) / 60:.0f} minutes")


Config: 200 titles x up to 5 pages (~1000 requests, ~1s apart)
Estimated runtime: ~33 minutes


## 2. Step 1 — Build the title index

Titles come from MAL's **Top Anime by Popularity** ranking (`/topanime.php?type=bypopularity`). The first
plan used the advanced-search URL, but that returns newly released titles, most with no reviews yet.
Popular titles hold thousands of reviews each and span many genres.

`fetch_html()` wraps every request with browser headers (MAL's firewall blocks bare clients), a timeout,
back-off on 403/429 and the disk cache, and returns `None` on permanent failure so one bad page never stops
the run.

In [2]:
# ============================================================
# Step 1 — fetch helper + title index
# ============================================================
session = requests.Session()   # one session = connection re-use = fewer TCP handshakes

def polite_sleep():
    """Base delay + random jitter so request timing looks like a human, not a metronome.
    Sustained rate stays under 1 request/second (see rate note in section 1)."""
    time.sleep(REQUEST_DELAY + random.uniform(0.0, 0.5))

def _cache_path(url):
    return os.path.join(CACHE_DIR, hashlib.md5(url.encode("utf-8")).hexdigest() + ".html")

def fetch_html(url, max_retries=3):
    """Return the HTML of a URL, or None if it permanently fails.

    A local disk cache sits in front of the network: pages already fetched in a
    previous (or crashed) run are read from data/cache/ instantly, so re-running
    this notebook never re-downloads what the server already gave us. Delete the
    cache folder to force a completely fresh scrape."""
    cpath = _cache_path(url)
    if os.path.exists(cpath):
        with open(cpath, encoding="utf-8") as f:
            return f.read()

    for attempt in range(1, max_retries + 1):
        try:
            resp = session.get(url, headers=HEADERS, timeout=30)
            if resp.status_code == 200:
                with open(cpath, "w", encoding="utf-8") as f:
                    f.write(resp.text)
                polite_sleep()                           # only sleep after real network hits
                return resp.text
            if resp.status_code in (403, 429):           # WAF push-back -> back off harder
                wait = max(REQUEST_DELAY, 3) * attempt * 2
                print(f"  [!] HTTP {resp.status_code} -> backing off {wait}s ({url})")
                time.sleep(wait)
            elif resp.status_code == 404:                 # page simply doesn't exist
                return None
            else:
                print(f"  [!] HTTP {resp.status_code} on {url} (attempt {attempt})")
                time.sleep(REQUEST_DELAY * attempt)
        except requests.RequestException as exc:
            print(f"  [!] {type(exc).__name__}: {exc} (attempt {attempt})")
            time.sleep(REQUEST_DELAY * attempt)
    return None

ANIME_LINK_RE = re.compile(r"https://myanimelist\.net/anime/(\d+)/([^/?]+)")

def get_top_anime(n_titles):
    """Walk the 'Top Anime by Popularity' ranking and return n_titles dicts of
    {anime_id, slug, title}."""
    collected, offset = [], 0
    while len(collected) < n_titles:
        url = f"{BASE_URL}/topanime.php?type=bypopularity&limit={offset}"
        html = fetch_html(url)
        if html is None:
            print(f"  [!] Could not fetch ranking offset {offset} - stopping index build.")
            break
        soup = BeautifulSoup(html, "lxml")
        rows = soup.select("tr.ranking-list")
        if not rows:
            break                                        # ran past the end of the ranking
        for row in rows:
            link = row.select_one("h3 a")
            m = ANIME_LINK_RE.match(link["href"]) if link else None
            if m:
                collected.append({"anime_id": int(m.group(1)),
                                  "slug": m.group(2),
                                  "title": link.get_text(strip=True)})
        offset += 50
    return collected[:n_titles]

anime_index = get_top_anime(N_TITLES)
print(f"Collected {len(anime_index)} titles for the scrape index.")
pd.DataFrame(anime_index).head(10)


Collected 200 titles for the scrape index.


,anime_id,slug,title
0,16498,Shingeki_no_Kyojin,Shingeki no Kyojin
1,1535,Death_Note,Death Note
2,5114,Fullmetal_Alchemist__Brotherhood,Fullmetal Alchemist: Brotherhood
3,30276,One_Punch_Man,One Punch Man
4,38000,Kimetsu_no_Yaiba,Kimetsu no Yaiba
5,31964,Boku_no_Hero_Academia,Boku no Hero Academia
6,11757,Sword_Art_Online,Sword Art Online
7,11061,Hunter_x_Hunter_2011,Hunter x Hunter (2011)
8,20,Naruto,Naruto
9,40748,Jujutsu_Kaisen,Jujutsu Kaisen


## 3. Step 2 — Parse a review page

Reviews live at `/anime/<id>/<slug>/reviews?p=<page>`, 20 per page and server-rendered, so `requests` +
BeautifulSoup is enough (no Selenium).

| Field | CSS selector |
|---|---|
| Review block | `div.review-element` |
| Star rating | `div.rating span.num` |
| Review text | `div.text` |
| Verdict tag | `div.tags` |
| Post date | `div.update_at` |

Two details matter for data quality:
1. **No truncation.** The "read more" text is already in the HTML, just hidden with CSS, so `get_text()`
   recovers the full review.
2. **No label leakage.** The text container also holds the rating widget. It is removed with `.decompose()`
   first, otherwise every review would contain *"Reviewer's Rating: 10"* and the model could cheat.

Reviews missing a rating or text are discarded at parse time.

In [3]:
# ============================================================
# Step 2 — review-page parser
# ============================================================
WHITESPACE_RE = re.compile(r"\s+")

def parse_review(elem, anime_id, anime_title):
    """Turn one <div class='review-element'> into a flat record (or None if unusable)."""
    rating_el = elem.select_one("div.rating span.num")
    text_el   = elem.select_one("div.text")
    if rating_el is None or text_el is None:
        return None                                     # completeness rule: need BOTH fields

    try:
        rating = int(rating_el.get_text(strip=True))
    except ValueError:
        return None
    if not 1 <= rating <= 10:
        return None

    tag_el  = elem.select_one("div.tags")
    date_el = elem.select_one("div.update_at")

    # Strip nested widgets so the text column contains ONLY the review prose
    # (prevents 'Reviewer's Rating: 10' leaking the label into the feature text).
    # Note: including "a" here also deletes any hyperlinked words inside the body.
    # Accepted trade-off - MAL links are almost always spoiler tags or title
    # cross-links, so losing a few words costs less than keeping widget markup.
    for junk in text_el.select("div.rating, div.tags, a"):
        junk.decompose()

    text = WHITESPACE_RE.sub(" ", text_el.get_text(" ", strip=True)).strip()
    if len(text) < 20:                                  # drop empty/placeholder bodies
        return None

    return {
        "anime_id":    anime_id,
        "anime_title": anime_title,
        "rating":      rating,
        "tag":         tag_el.get_text(" ", strip=True) if tag_el else None,
        "review_date": date_el.get_text(strip=True) if date_el else None,
        "review_text": text,
    }

def scrape_title_reviews(anime, max_pages):
    """Scrape up to max_pages of reviews for one title. Returns a list of records."""
    records = []
    for page in range(1, max_pages + 1):
        url = f"{BASE_URL}/anime/{anime['anime_id']}/{anime['slug']}/reviews?p={page}"
        html = fetch_html(url)
        if html is None:
            break
        soup  = BeautifulSoup(html, "lxml")
        elems = soup.select("div.review-element")
        if not elems:
            break                                       # no reviews on this page -> done
        for elem in elems:
            rec = parse_review(elem, anime["anime_id"], anime["title"])
            if rec is not None:
                records.append(rec)
        if len(elems) < 20:
            break                                       # short page = last page of reviews
    return records

# quick smoke test on the most popular title before committing to the full loop
_sample = scrape_title_reviews(anime_index[0], max_pages=1)
print(f"Smoke test on '{anime_index[0]['title']}': {len(_sample)} reviews parsed")
pd.DataFrame(_sample).head(3)


Smoke test on 'Shingeki no Kyojin': 20 reviews parsed


,anime_id,anime_title,rating,tag,review_date,review_text
0,16498,Shingeki no Kyojin,10,Recommended,"Oct 2, 2014","Oh dear Shingeki no Kyojin, where do I even be..."
1,16498,Shingeki no Kyojin,10,Recommended,"May 7, 2014",I started to follow the manga after watching t...
2,16498,Shingeki no Kyojin,10,Recommended,"Aug 8, 2014","In the 80's, Mobile Suit Gundam catapulted ani..."


## 4. Step 3 — Main scraping loop

Up to `MAX_REVIEW_PAGES` pages per title, so at most 100 reviews per title. The cap is deliberate: each show
gets a roughly equal share and no single fandom dominates, at the cost of leaving most of each popular
title's reviews unscraped (breadth over depth). The loop checkpoints to `data/_scrape_checkpoint.csv` every
10 titles and skips any title that fails instead of stopping.

In [4]:
# ============================================================
# Step 3 — harvest loop (this is the long-running cell)
# ============================================================
start_time  = time.time()
all_reviews = []

for i, anime in enumerate(anime_index, start=1):
    records = scrape_title_reviews(anime, MAX_REVIEW_PAGES)
    all_reviews.extend(records)
    print(f"[{i:>2}/{len(anime_index)}] {anime['title'][:45]:<45} "
          f"+{len(records):>3} reviews (total {len(all_reviews)})")

    if i % 10 == 0:                                    # checkpoint every 10 titles
        pd.DataFrame(all_reviews).to_csv(
            os.path.join(DATA_DIR, "_scrape_checkpoint.csv"),
            index=False, encoding="utf-8-sig")

elapsed = (time.time() - start_time) / 60
print(f"\nScrape complete: {len(all_reviews)} raw reviews "
      f"from {len(anime_index)} titles in {elapsed:.1f} min")


[ 1/200] Shingeki no Kyojin                            +100 reviews (total 100)


[ 2/200] Death Note                                    +100 reviews (total 200)


[ 3/200] Fullmetal Alchemist: Brotherhood              +100 reviews (total 300)


[ 4/200] One Punch Man                                 +100 reviews (total 400)


[ 5/200] Kimetsu no Yaiba                              +100 reviews (total 500)


...

[193/200] Yakusoku no Neverland 2nd Season              +100 reviews (total 19023)
[194/200] Dandadan                                      +100 reviews (total 19123)


[195/200] Gekkan Shoujo Nozaki-kun                      +100 reviews (total 19223)
[196/200] Tokyo Ghoul:re 2nd Season                     +100 reviews (total 19323)


[197/200] Kimetsu no Yaiba: Mugen Ressha-hen            +100 reviews (total 19423)
[198/200] 86                                            +100 reviews (total 19523)


[199/200] Nichijou                                      +100 reviews (total 19623)
[200/200] Suzumiya Haruhi no Yuuutsu                    +100 reviews (total 19723)



Scrape complete: 19723 raw reviews from 200 titles in 0.7 min


## 5. Step 4 — Verify the raw scrape

Read-only checks; nothing is removed or changed here.
- **Is it structurally valid?** Row count, columns, dtypes, no missing ratings or text, ratings within 1–10,
  duplicates counted (they are removed in notebook 2).
- **Is it fit for purpose?** Enough low ratings to form a negative class, reviews long enough for
  bag-of-words, and whether the 100-review cap was reached.

In [5]:
# ============================================================
# Step 4 — raw verification (READ-ONLY: nothing is modified or removed)
# ============================================================
df = pd.DataFrame(all_reviews)

# ---- A. structural validity -------------------------------------------------
print(f"Raw shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("Columns:", list(df.columns))

print("\nDtypes:")
print(df.dtypes)

print("\nNull counts per column:")
print(df.isna().sum())

# validity: the parser should have rejected anything outside 1-10
print(f"\nRating range: {df['rating'].min()} to {df['rating'].max()} "
      f"(expected 1 to 10) -> {'OK' if df['rating'].between(1, 10).all() else 'PROBLEM'}")

# completeness: both modelling-critical fields must always be present
missing_core = df[["rating", "review_text"]].isna().any(axis=1).sum()
print(f"Records missing rating or text: {missing_core} "
      f"-> {'OK' if missing_core == 0 else 'PROBLEM'}")

# duplicates: REPORTED ONLY - removal is a cleaning step, deferred to notebook 2
dupes = df.duplicated(subset=["anime_id", "review_text"]).sum()
print(f"\nDuplicate (anime_id, review_text) pairs found: {dupes} "
      f"({dupes/len(df)*100:.2f}%)  <- left in place, notebook 2 removes these")

# ---- B. fitness for purpose -------------------------------------------------
print("\n" + "=" * 62)
print("RATING DISTRIBUTION (raw 1-10 scale)")
print("=" * 62)
dist = df["rating"].value_counts().sort_index()
for stars, n in dist.items():
    band = "neg " if stars <= 4 else ("neut" if stars <= 6 else "pos ")
    print(f"  {stars:>2} stars [{band}] {n:>6,}  {n/len(df)*100:5.1f}%  {'#' * round(n/len(df)*100)}")

future_neg = df["rating"].between(1, 4).sum()
future_neu = df["rating"].between(5, 6).sum()
future_pos = df["rating"].between(7, 10).sum()
print(f"\n  Future negative class (1-4): {future_neg:>6,} ({future_neg/len(df)*100:4.1f}%)")
print(f"  Neutral, to be dropped (5-6): {future_neu:>5,} ({future_neu/len(df)*100:4.1f}%)")
print(f"  Future positive class (7-10): {future_pos:>5,} ({future_pos/len(df)*100:4.1f}%)")
print(f"  -> post-label class ratio approx {future_pos/future_neg:.1f} : 1 (positive : negative)")

# review length: is this real prose?
wc = df["review_text"].str.split().str.len()
print("\n" + "=" * 62)
print("REVIEW LENGTH (words, raw text)")
print("=" * 62)
print(f"  min {wc.min():,} | 25th {wc.quantile(.25):,.0f} | median {wc.median():,.0f} | "
      f"75th {wc.quantile(.75):,.0f} | max {wc.max():,}")
print(f"  Reviews under 50 words: {(wc < 50).sum():,} ({(wc < 50).mean()*100:.1f}%)")
print(f"  Reviews over 1,000 words: {(wc > 1000).sum():,} ({(wc > 1000).mean()*100:.1f}%)")

# coverage + did the per-title cap bind?
per_title = df.groupby("anime_title").size()
cap = MAX_REVIEW_PAGES * 20
capped = (per_title >= cap).sum()
print("\n" + "=" * 62)
print("COVERAGE")
print("=" * 62)
print(f"  Titles represented: {per_title.size} | reviews per title: "
      f"min {per_title.min()}, median {per_title.median():.0f}, max {per_title.max()}")
print(f"  Titles that hit the {cap}-review cap: {capped} of {per_title.size} "
      f"({capped/per_title.size*100:.0f}%)")
print(f"  -> those titles have MORE reviews available than we collected "
      f"(breadth-over-depth choice, see Step 3)")


Raw shape: 19,723 rows x 6 columns
Columns: ['anime_id', 'anime_title', 'rating', 'tag', 'review_date', 'review_text']

Dtypes:
anime_id       int64
anime_title      str
rating         int64
tag              str
review_date      str
review_text      str
dtype: object

Null counts per column:
anime_id       0
anime_title    0
rating         0
tag            0
review_date    0
review_text    0
dtype: int64

Rating range: 1 to 10 (expected 1 to 10) -> OK
Records missing rating or text: 0 -> OK

Duplicate (anime_id, review_text) pairs found: 42 (0.21%)  <- left in place, notebook 2 removes these

RATING DISTRIBUTION (raw 1-10 scale)
   1 stars [neg ]    553    2.8%  ###
   2 stars [neg ]    542    2.7%  ###
   3 stars [neg ]  1,085    5.5%  ######
   4 stars [neg ]  1,130    5.7%  ######
   5 stars [neut]  1,322    6.7%  #######
   6 stars [neut]  1,512    7.7%  ########
   7 stars [pos ]  1,985   10.1%  ##########
   8 stars [pos ]  2,802   14.2%  ##############
   9 stars [pos ]  3,929  


REVIEW LENGTH (words, raw text)
  min 12 | 25th 207 | median 395 | 75th 729 | max 11,544
  Reviews under 50 words: 22 (0.1%)
  Reviews over 1,000 words: 2,889 (14.6%)

COVERAGE
  Titles represented: 200 | reviews per title: min 48, median 100, max 100
  Titles that hit the 100-review cap: 188 of 200 (94%)
  -> those titles have MORE reviews available than we collected (breadth-over-depth choice, see Step 3)


### Findings

- **Structurally clean.** 19,723 rows from all 200 titles, zero missing values, every rating in range. The
  42 duplicates (0.2%) come from MAL's live pagination and are left for notebook 2 to remove.
- **Imbalanced but usable.** Roughly 4 : 1 positive to negative after labelling, with ~3,300 reviews in the
  1–4 band. That is enough to learn from, but **accuracy will be misleading** (always guessing "positive"
  scores ~80%), so modelling needs per-class metrics and class weighting. The skew is genuine: people who
  finish and review popular shows tend to have liked them.
- **Long-form text.** The median review runs to several hundred words, which suits bag-of-words and TF-IDF.
- **Even coverage.** Almost every title reached the 100-review cap, so no show dominates the corpus.

## 6. Step 5 — Save the corpus and split 95/5

Dataset B (5%) is held out for the final test and never used during development, so its score reflects
genuinely unseen data. The 5% is sampled **within each star rating**, so B keeps the same rating mix as the
full corpus; a plain random 5% could end up with too few negative reviews to measure. A fixed
`random_state` makes the split reproducible.

In [6]:
# ============================================================
# Step 5 — save combined corpus, then stratified 95/5 split
# ============================================================
final_cols = ["anime_id", "anime_title", "rating", "tag", "review_date", "review_text"]
corpus = df[final_cols].copy()

combined_path = os.path.join(DATA_DIR, "raw_reviews_combined.csv")
corpus.to_csv(combined_path, index=False, encoding="utf-8-sig")
print(f"Combined corpus  : {combined_path}  ({len(corpus)} rows)")

# stratified 5% sample within every rating value -> Dataset B
dataset_b = (corpus.groupby("rating", group_keys=False)
                   .sample(frac=0.05, random_state=RANDOM_STATE))
dataset_a = corpus.drop(dataset_b.index)

a_path = os.path.join(DATA_DIR, "Dataset_A_raw.csv")
b_path = os.path.join(DATA_DIR, "Dataset_B_raw.csv")
dataset_a.to_csv(a_path, index=False, encoding="utf-8-sig")
dataset_b.to_csv(b_path, index=False, encoding="utf-8-sig")

print(f"Dataset A (train): {a_path}  ({len(dataset_a)} rows, "
      f"{len(dataset_a)/len(corpus)*100:.1f}%)")
print(f"Dataset B (test) : {b_path}  ({len(dataset_b)} rows, "
      f"{len(dataset_b)/len(corpus)*100:.1f}%)")

# verify the stratification actually preserved the rating profile
check = pd.DataFrame({
    "combined %": corpus["rating"].value_counts(normalize=True).sort_index().round(3) * 100,
    "Dataset A %": dataset_a["rating"].value_counts(normalize=True).sort_index().round(3) * 100,
    "Dataset B %": dataset_b["rating"].value_counts(normalize=True).sort_index().round(3) * 100,
})
check


Combined corpus  : data\raw_reviews_combined.csv  (19723 rows)


Dataset A (train): data\Dataset_A_raw.csv  (18738 rows, 95.0%)
Dataset B (test) : data\Dataset_B_raw.csv  (985 rows, 5.0%)


,combined %,Dataset A %,Dataset B %
rating,,,
1,2.8,2.8,2.8
2,2.7,2.7,2.7
3,5.5,5.5,5.5
4,5.7,5.7,5.7
5,6.7,6.7,6.7
6,7.7,7.7,7.7
7,10.1,10.1,10.1
8,14.2,14.2,14.2
9,19.9,19.9,19.9


## 7. Summary

| | |
|---|---|
| Corpus | 19,723 raw reviews across 200 titles, verified, duplicates left in |
| Split | Dataset A 18,738 rows / Dataset B 985 rows, rating-stratified |

**Carried into notebook 2:** the ~4 : 1 imbalance (use per-class metrics and class weights); show and
character names throughout the text (must be stripped for the model to generalise); 42 duplicates (remove
before splitting).

**Next:** `02_Data_Preparation.ipynb`